In [0]:
file_path = "/Volumes/workspace/energyflow/pjm_source/PJME_hourly.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

display(df.limit(10))

In [0]:
print("Total rows:", df.count())
print("Columns:", df.columns)

In [0]:
import os

print("Current directory:", os.getcwd())

In [0]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

sys.path.insert(0, str(src_path))

print("Python source path:", src_path)
print("Exists:", src_path.exists())

In [0]:
from energyflow.config import load_config

config = load_config()

print("Source configuration:")
print(config["data"]["source_file"])

In [0]:
catalog = config["databricks"]["catalog"]
schema = config["databricks"]["schema"]
source = config["data"]["source_file"]

source_path = (
    f"/Volumes/{catalog}/{schema}/"
    f"{source['volume']}/{source['filename']}"
)

print(source_path)

In [0]:
from energyflow.config import load_config
from energyflow.ingestion.ingest_pjm import ingest_pjm

config = load_config()

row_count = ingest_pjm(spark, config)

print(f"Ingested rows: {row_count:,}")

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM workspace.energyflow.pjm_raw;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN datetime_raw IS NULL THEN 1 ELSE 0 END)
        AS missing_timestamps,
    SUM(CASE WHEN demand_mw IS NULL THEN 1 ELSE 0 END)
        AS missing_demand,
    MIN(demand_mw) AS min_demand,
    MAX(demand_mw) AS max_demand
FROM workspace.energyflow.pjm_raw;

In [0]:
%sql
SELECT
    datetime_raw,
    COUNT(*) AS occurrences
FROM workspace.energyflow.pjm_raw
GROUP BY datetime_raw
HAVING COUNT(*) > 1
ORDER BY occurrences DESC;

In [0]:
%sql
SELECT
    datetime_raw,
    demand_mw,
    COUNT(*) OVER (
        PARTITION BY datetime_raw
    ) AS occurrences
FROM workspace.energyflow.pjm_raw
WHERE datetime_raw BETWEEN '2017-11-05 00:00:00'
                       AND '2017-11-05 06:00:00'
ORDER BY datetime_raw, demand_mw;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(
        TRY_CAST(datetime_raw AS TIMESTAMP_NTZ) IS NULL
    ) AS invalid_timestamps,
    MIN(TRY_CAST(datetime_raw AS TIMESTAMP_NTZ)) AS earliest_timestamp,
    MAX(TRY_CAST(datetime_raw AS TIMESTAMP_NTZ)) AS latest_timestamp
FROM workspace.energyflow.pjm_raw;

In [0]:
from energyflow.config import load_config, get_table_name
from energyflow.processing.validate import validate_pjm

config = load_config()

raw_table = get_table_name(config, "raw")
df = spark.table(raw_table)

report = validate_pjm(df)

for metric, value in report.items():
    print(f"{metric}: {value}")

In [0]:
%sql
WITH ordered AS (
    SELECT
        datetime_raw,
        LAG(
            TRY_CAST(datetime_raw AS TIMESTAMP_NTZ)
        ) OVER (
            ORDER BY TRY_CAST(datetime_raw AS TIMESTAMP_NTZ)
        ) AS previous_timestamp
    FROM (
        SELECT DISTINCT datetime_raw
        FROM workspace.energyflow.pjm_raw
    )
),

gaps AS (
    SELECT
        previous_timestamp,
        TRY_CAST(datetime_raw AS TIMESTAMP_NTZ)
            AS current_timestamp,
        TIMESTAMPDIFF(
            HOUR,
            previous_timestamp,
            TRY_CAST(datetime_raw AS TIMESTAMP_NTZ)
        ) AS gap_hours
    FROM ordered
)

SELECT
    previous_timestamp,
    current_timestamp,
    gap_hours,
    gap_hours - 1 AS missing_clock_hours
FROM gaps
WHERE gap_hours > 1
ORDER BY previous_timestamp;

In [0]:
from energyflow.config import load_config
from energyflow.processing.clean import save_clean_data

config = load_config()

row_count = save_clean_data(spark, config)

print(f"Cleaned rows: {row_count:,}")

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(is_duplicate_hour) AS flagged_rows,
    COUNT(DISTINCT datetime_local) AS unique_timestamps,
    MIN(datetime_local) AS earliest_timestamp,
    MAX(datetime_local) AS latest_timestamp
FROM workspace.energyflow.pjm_clean;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(is_duplicate_hour) AS flagged_rows,
    COUNT(DISTINCT datetime_local) AS unique_timestamps
FROM workspace.energyflow.pjm_clean;

In [0]:
from energyflow.config import load_config
from energyflow.processing.features import save_features

config = load_config()

row_count = save_features(spark, config)

print(f"Feature rows: {row_count:,}")

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT segment_id) AS total_segments,
    COUNT_IF(has_full_history) AS model_ready_rows,
    COUNT_IF(lag_168 IS NULL) AS missing_lag_168,
    MIN(datetime_local) AS earliest_timestamp,
    MAX(datetime_local) AS latest_timestamp
FROM workspace.energyflow.pjm_features;

In [0]:
%sql
SELECT
    datetime_local,
    segment_id,
    demand_mw,
    lag_1,
    lag_24,
    history_count,
    has_full_history
FROM workspace.energyflow.pjm_features
WHERE datetime_local BETWEEN
    TIMESTAMP_NTZ '2010-12-09 21:00:00'
    AND TIMESTAMP_NTZ '2010-12-10 04:00:00'
ORDER BY datetime_local;

In [0]:
from energyflow.config import load_config
from energyflow.pipeline import run_pipeline

config = load_config()

run_pipeline(spark, config)

In [0]:
%sql
SELECT 'raw' AS stage, COUNT(*) AS total_rows
FROM workspace.energyflow.pjm_raw

UNION ALL

SELECT 'clean', COUNT(*)
FROM workspace.energyflow.pjm_clean

UNION ALL

SELECT 'features', COUNT(*)
FROM workspace.energyflow.pjm_features;

In [0]:
import mlflow
from pathlib import Path

run_id = "b28297cc1aad41adb4e96854c23c2e18"

artifact_dir = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="lstm_artifacts"
)

print("Downloaded artifacts:")
for file in Path(artifact_dir).iterdir():
    print(file.name)

In [0]:
import json
import torch

from energyflow.training.lstm_model import EnergyLSTM

artifact_dir = Path(artifact_dir)

model_config = json.loads(
    (artifact_dir / "model_config.json").read_text()
)

scaler_config = json.loads(
    (artifact_dir / "scaler.json").read_text()
)

model = EnergyLSTM(
    input_size=model_config["input_size"],
    hidden_size=model_config["hidden_size"],
    num_layers=model_config["num_layers"],
    dropout=model_config["dropout"],
    forecast_horizon=model_config["forecast_horizon"],
)

state_dict = torch.load(
    artifact_dir / "lstm_weights.pth",
    map_location="cpu",
    weights_only=True,
)

model.load_state_dict(state_dict)
model.eval()

print("LSTM loaded successfully.")
print("Lookback:", model_config["lookback"])
print("Forecast horizon:", model_config["forecast_horizon"])
print("Scaler mean:", scaler_config["mean"])
print("Scaler std:", scaler_config["std"])

In [0]:
%sql
SELECT
    COUNT(*) AS total_predictions,
    COUNT(DISTINCT forecast_origin) AS forecast_windows,
    MIN(forecast_timestamp) AS first_forecast,
    MAX(forecast_timestamp) AS last_forecast,
    ROUND(AVG(absolute_error_mw), 2) AS mae_mw,
    SQRT(AVG(POWER(error_mw, 2))) AS rmse_mw
FROM workspace.energyflow.pjm_predictions;

In [0]:
%sql
SELECT
    forecast_origin,
    forecast_timestamp,
    horizon_hour,
    ROUND(predicted_mw, 2) AS predicted_mw,
    ROUND(actual_mw, 2) AS actual_mw,
    ROUND(absolute_error_mw, 2) AS absolute_error_mw
FROM workspace.energyflow.pjm_predictions
ORDER BY forecast_origin, horizon_hour
LIMIT 24;